# AACAgent — GPU Evaluation Notebook (HuggingFace / Colab)

Evaluation on GPU using `HuggingFaceBackend` with HuggingFace Transformers models.
Designed to run on **Google Colab with GPU runtime** (T4 or higher),
or on the SLURM cluster via `eval/cluster_work/run_eval_cluster.sh`.

The backend is `HuggingFaceBackend` from `agent/backends.py`.
The CSV output format is identical to `eval_cpu.ipynb` — results are
directly comparable with llama.cpp CPU runs.

**Colab instructions:**
1. Runtime → Change runtime type → **T4 GPU** (or A100 if available)
2. Edit the `colab-env` cell with the desired parameters
3. Run all cells in order

**CSV columns:**
```
row_idx, input_type, turn_pos, concept_text,
called_get_time, called_get_schedule,
predicted_ids, plan_method, resolve_method, planner_concepts
```

In [ ]:
# ─── COLAB ONLY — clone repo ──────────────────────────────────────────────────
import subprocess, sys, os

subprocess.run(
    ["git", "clone", "https://github.com/lollopelle01/aac-mcp-agent.git",
     "/content/aac-mcp-agent"],
    check=True,
)

PROJECT_ROOT = "/content/aac-mcp-agent"
os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, "app", "src"))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "app"))

print("Repo cloned. CWD:", os.getcwd())

In [ ]:
# ─── ENV VARS — edit here before running ─────────────────────────────────────
# NB_MODELS: space-separated HF repos
#   e.g. "Qwen/Qwen2.5-3B-Instruct" or "Qwen/Qwen2.5-3B-Instruct meta-llama/Llama-3.2-3B-Instruct"
# NB_LOAD_8BIT: "True" for INT8 via bitsandbytes (~50% less VRAM)
# NB_SPLIT_FILTER: "clear" | "vague" | "all"
import os

os.environ["NB_MODELS"]            = "Qwen/Qwen2.5-3B-Instruct"
os.environ["NB_N_ROWS"]            = "100"   # 0 = all 1760 sentences
os.environ["NB_LANG"]              = "en_eval"
os.environ["NB_MAX_RESULTS"]       = "0"     # 0 = use default from settings.py (25)
os.environ["NB_SEED"]              = "42"
os.environ["NB_SPLIT_FILTER"]      = "all"   # "clear" | "vague" | "all"
os.environ["NB_MAX_NEW_TOKENS"]    = "150"
os.environ["NB_LOAD_8BIT"]         = "True"  # recommended on T4 (16 GB)
os.environ["NB_DTYPE"]             = "float16"
os.environ["NB_OUTPUT_CSV"]        = "/content/aac-mcp-agent/eval/gpu/eval_gpu_colab.csv"
os.environ["NB_ANNOTATED_PARQUET"] = "/content/aac-mcp-agent/annotation/eval_final.parquet"

In [ ]:
import os

MODELS_RAW        = os.environ.get("NB_MODELS",            "Qwen/Qwen2.5-3B-Instruct")
N_ROWS_ENV        = os.environ.get("NB_N_ROWS",             "100")
LANG_CODE         = os.environ.get("NB_LANG",               "en_eval")
_max_results_env  = int(os.environ.get("NB_MAX_RESULTS",    "0"))
SEED              = int(os.environ.get("NB_SEED",            "42"))
SPLIT_FILTER      = os.environ.get("NB_SPLIT_FILTER",        "all")
MAX_NEW_TOKENS    = int(os.environ.get("NB_MAX_NEW_TOKENS",  "150"))
LOAD_8BIT_ENV     = os.environ.get("NB_LOAD_8BIT",           "True")
DTYPE             = os.environ.get("NB_DTYPE",               "float16")
ANNOTATED_PARQUET = os.environ.get("NB_ANNOTATED_PARQUET",
                                   "/content/aac-mcp-agent/annotation/eval_final.parquet")
OUTPUT_CSV        = os.environ.get("NB_OUTPUT_CSV",
                                   "/content/aac-mcp-agent/eval/gpu/eval_gpu_colab.csv")

MODELS    = MODELS_RAW.split()
N_ROWS    = int(N_ROWS_ENV)
LOAD_8BIT = LOAD_8BIT_ENV.lower() == "true"

print(f"Models            : {MODELS}")
print(f"N_rows            : {N_ROWS if N_ROWS > 0 else 'full dataset (1760)'}")
print(f"Seed              : {SEED}")
print(f"Lang              : {LANG_CODE}")
print(f"Max results       : {_max_results_env if _max_results_env > 0 else 'default (settings.py)'}")
print(f"Split filter      : {SPLIT_FILTER}")
print(f"Max new tokens    : {MAX_NEW_TOKENS}")
print(f"Load 8-bit        : {LOAD_8BIT}")
print(f"Dtype             : {DTYPE}")
print(f"Annotated parquet : {ANNOTATED_PARQUET}")
print(f"Output CSV        : {OUTPUT_CSV}")

In [ ]:
import subprocess, sys

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

_pip(
    "transformers>=4.40,<5.0",
    "accelerate>=0.29",
    "datasets>=2.19",
    "pandas>=2.0",
    "pyarrow>=14",
    "huggingface_hub>=0.22",
    "tqdm>=4.66",
    "sentencepiece>=0.1.99",
    "protobuf>=3.20",
    "spacy>=3.7",
    "fastmcp",
    "pydantic>=2.0",
    "httpx>=0.24",
    "python-dotenv",
)

if LOAD_8BIT:
    _pip("bitsandbytes>=0.43")

# spaCy model
try:
    import spacy
    spacy.load("en_core_web_sm")
    print("spaCy model en_core_web_sm already present.")
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"])

print("\nAll packages ready.")

In [ ]:
# ─── GPU check — the run only makes sense with a GPU available ────────────────
import torch

if not torch.cuda.is_available():
    print("⚠️  WARNING: no GPU detected.")
    print("   This notebook is designed to run on GPU.")
    print("   On Colab: Runtime → Change runtime type → T4 GPU")
    print("   Execution will continue on CPU but will be very slow.")
else:
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  ({p.total_memory / 1e9:.1f} GB VRAM)")
    print(f"  CUDA version : {torch.version.cuda}")
    if LOAD_8BIT and torch.cuda.get_device_properties(0).total_memory < 12e9:
        print("  ⚠️  <12 GB VRAM — NB_LOAD_8BIT=True recommended (already set).")

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/aac-mcp-agent")
APP          = PROJECT_ROOT / "app"
SRC          = APP / "src"

for p in [str(SRC), str(APP)]:
    if p not in sys.path:
        sys.path.insert(0, p)

EVAL_PARQUET = Path(ANNOTATED_PARQUET)

_out = Path(OUTPUT_CSV)
_out.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV_PATH = _out

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"Eval parquet  : {EVAL_PARQUET}  exists={EVAL_PARQUET.exists()}")
print(f"Output CSV    : {OUTPUT_CSV_PATH}")

In [ ]:
import ast
import csv
import logging
import time
from datetime import timedelta

import pandas as pd
import torch

# ── Logging ───────────────────────────────────────────────────────────────────
try:
    from logs.logging_config import setup_logging
    setup_logging()
except Exception:
    logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)

# ── Project imports ───────────────────────────────────────────────────────────
from config import AGENT_MAX_RESULTS
from agent.agent import AACAgent, EvalContext
from agent.backends import HuggingFaceBackend
from agent.session import SessionMemory
from mcp_server.models import Pictogram, Keyword
from mcp_server.tools.arasaac import get_pictogram_metadata
import mcp_server.tools.arasaac as _arasaac_mod

# Force the frozen dataset for all ARASAAC lookups
_arasaac_mod.LANG = LANG_CODE  # type: ignore[attr-defined]

EVAL_MAX_RESULTS = _max_results_env if _max_results_env > 0 else AGENT_MAX_RESULTS

print("Imports OK.")
print(f"CUDA available   : {torch.cuda.is_available()}")
print(f"EVAL_MAX_RESULTS : {EVAL_MAX_RESULTS}")

In [ ]:
# CSV columns — identical to eval_cpu.ipynb for direct comparability
CSV_COLUMNS = [
    "row_idx",
    "input_type",
    "turn_pos",
    "concept_text",
    "called_get_time",
    "called_get_schedule",
    "predicted_ids",
    "plan_method",
    "resolve_method",
    "planner_concepts",
]

In [ ]:
print(f"Loading dataset from {EVAL_PARQUET} ...")
df_full = pd.read_parquet(EVAL_PARQUET)
print(f"Full dataset : {len(df_full):,} rows × {df_full.shape[1]} columns")
print(f"Columns      : {list(df_full.columns)}")

if df_full["concepts"].dtype == object and isinstance(df_full["concepts"].iloc[0], str):
    df_full["concepts"] = df_full["concepts"].apply(ast.literal_eval)

if df_full["schedule"].dtype == object and isinstance(df_full["schedule"].iloc[0], str):
    df_full["schedule"] = df_full["schedule"].apply(ast.literal_eval)

assert set(["sentence", "concepts", "caregiver_clear", "caregiver_vague",
            "time_of_day", "event_time", "schedule", "split"]).issubset(df_full.columns), (
    f"Missing columns. Found: {list(df_full.columns)}"
)
assert df_full["concepts"].iloc[0][0].get("concept_text") is not None
print(f"Sanity OK — example concept: {df_full['concepts'].iloc[0][0]}")
print(f"Split distribution:\n{df_full['split'].value_counts()}")

# Sampling
# On Colab with 100 rows, GPU warmup + inference takes ~15–20 minutes.
# For a full run on 1760 sentences use NB_N_ROWS=0 on the cluster.
COLAB_SAMPLE = 200
_cap = N_ROWS if N_ROWS > 0 else COLAB_SAMPLE
df   = df_full.sample(min(_cap, len(df_full)), random_state=SEED).reset_index(drop=True)
print(f"\nEffective sample: {len(df)} rows (cap={_cap}, seed={SEED})")

In [ ]:
# ── Helper: gold metadata ─────────────────────────────────────────────────────

_gold_cache: dict[int, dict] = {}

def get_gold_meta(pic_id: int) -> dict:
    k = int(pic_id)
    if k not in _gold_cache:
        try:
            _gold_cache[k] = get_pictogram_metadata(pictogram_id=k, lang=LANG_CODE)
        except Exception:
            _gold_cache[k] = {}
    return _gold_cache[k]

def gold_as_pictogram(pic_id: int, concept: str) -> Pictogram:
    meta = get_gold_meta(pic_id)
    if meta:
        try:
            return Pictogram.model_validate(meta)
        except Exception:
            pass
    return Pictogram(id=int(pic_id), keywords=[Keyword(type=2, keyword=concept)])


# ── Helper: EvalContext ───────────────────────────────────────────────────────

def build_eval_ctx(row) -> EvalContext:
    """Build EvalContext with mock values from the annotated row.
    Used only at turn_pos==0."""
    mock_time = {
        "time_of_day": row["time_of_day"],
        "event_time":  str(row["event_time"]),
    }
    raw_sched = row["schedule"]
    if isinstance(raw_sched, str):
        raw_sched = ast.literal_eval(raw_sched)
    mock_schedule = raw_sched if isinstance(raw_sched, list) else []
    return EvalContext(mock_time=mock_time, mock_schedule=mock_schedule)


# ── Helper: teacher forcing ───────────────────────────────────────────────────

def teacher_force(agent: AACAgent, gold_id: int, concept: str) -> None:
    """Inject the gold pictogram into memory after each turn.
    Identical to eval_cpu.ipynb."""
    if not agent.memory.turns:
        return
    last = agent.memory.turns[-1]
    for t in last.topics:
        agent.memory.topic_frequency[t] = max(0, agent.memory.topic_frequency.get(t, 0) - 1)
    gold_pic    = gold_as_pictogram(gold_id, concept)
    gold_topics = SessionMemory.extract_topics([gold_pic])
    last.pictograms = [gold_pic]
    last.topics     = gold_topics
    for t in gold_topics:
        agent.memory.topic_frequency[t] = agent.memory.topic_frequency.get(t, 0) + 1


# ── Helper: incremental CSV ───────────────────────────────────────────────────

class IncrementalCSV:
    def __init__(self, path) -> None:
        self.path    = path
        self._is_new = not path.exists()
        self._buffer: list[dict] = []

    def add(self, rows: list[dict]) -> None:
        self._buffer.extend(rows)

    def flush(self) -> None:
        if not self._buffer:
            return
        mode = "w" if self._is_new else "a"
        with open(self.path, mode, newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS, extrasaction="ignore")
            if self._is_new:
                writer.writeheader()
                self._is_new = False
            writer.writerows(self._buffer)
        self._buffer.clear()


# ── Helper: progress tracker ──────────────────────────────────────────────────

class ProgressTracker:
    def __init__(self, total: int) -> None:
        self.total    = total
        self.n_done   = 0
        self.n_errors = 0
        self._t0      = time.monotonic()

    def record(self, row_results) -> None:
        self.n_done += 1

    def record_error(self) -> None:
        self.n_done  += 1
        self.n_errors += 1

    def print_progress(self) -> None:
        elapsed = time.monotonic() - self._t0
        avg_s   = elapsed / max(self.n_done, 1)
        eta_str = str(timedelta(seconds=int(avg_s * max(self.total - self.n_done, 0))))
        w = len(str(self.total))
        print(
            f"  [{self.n_done:>{w}}/{self.total}]"
            f"  ETA {eta_str}"
            f"  errors={self.n_errors}",
            flush=True,
        )


print("Helpers defined OK.")

In [ ]:
def run_multi_turn(agent: AACAgent, row: "pd.Series", input_type: str) -> list[dict]:
    """Run the multi-turn sequence for a row and an input_type.

    CONTEXT PROPAGATION:
    - Turn 0: real input (clear or vague) + FULL EvalContext with mock tools.
      If the planner decides call_tools=True (expected on vague), _collect_context()
      uses the mocks → time_of_day and schedule enter SessionMemory via Turn.
    - Turn 1+: input="" + EMPTY EvalContext.
      The planner reads prompt_summary() which includes time_of_day and the topics
      of the gold injected by teacher forcing from the previous turn.

    Teacher forcing: after each turn the gold pictogram is injected into
    memory so the next turn starts from the correct context.

    Identical to eval_cpu.ipynb — the logic does not depend on the LLM backend.
    """
    concepts = row["concepts"]
    caregiver_input_t0 = str(row["caregiver_clear" if input_type == "clear" else "caregiver_vague"])

    agent.reset_session()
    results: list[dict] = []

    for turn_pos, concept_entry in enumerate(concepts):
        concept_text = concept_entry["concept_text"]
        gold_id      = int(concept_entry["gold_id"])

        if turn_pos == 0:
            ec        = build_eval_ctx(row)
            raw_input = caregiver_input_t0
        else:
            ec        = EvalContext()
            raw_input = ""

        window = agent.run(raw_input, eval_ctx=ec)

        predicted_ids    = [p.id for p in window]
        planner_concepts = [e["concept"] for e in agent.last_resolve_info]
        resolve_method   = next(
            (e["method"] for e in agent.last_resolve_info if e["concept"] == concept_text),
            "none"
        )

        results.append({
            "row_idx":             row.name,
            "input_type":          input_type,
            "turn_pos":            turn_pos,
            "concept_text":        concept_text,
            "called_get_time":     "get_time"     in ec.tool_calls,
            "called_get_schedule": "get_schedule" in ec.tool_calls,
            "predicted_ids":       str(predicted_ids),
            "plan_method":         agent.last_plan_method,
            "resolve_method":      resolve_method,
            "planner_concepts":    str(planner_concepts),
        })

        teacher_force(agent, gold_id, concept_text)

    return results


print("run_multi_turn defined OK.")

In [ ]:
LOG_EVERY  = 10
SAVE_EVERY = 10

csv_writer = IncrementalCSV(OUTPUT_CSV_PATH)
total_t0   = time.monotonic()

for model_name in MODELS:
    print(f"\n{'━'*70}\n  MODEL: {model_name}\n{'━'*70}")

    backend = HuggingFaceBackend(
        model          = model_name,
        device         = "auto",
        load_in_8bit   = LOAD_8BIT,
        max_new_tokens = MAX_NEW_TOKENS,
        dtype          = DTYPE,
    )

    agent = AACAgent(
        model          = model_name,
        backend        = backend,
        lang           = LANG_CODE,
        max_results    = EVAL_MAX_RESULTS,
        fetch_schedule = False,   # mocked via EvalContext, no live calls
    )

    # Warm-up: load the model into VRAM before the loop (avoids latency on the first turn)
    print(f"  Loading model into VRAM ...", flush=True)
    t_load = time.monotonic()
    agent.backend._ensure_loaded()
    print(f"  Model loaded in {time.monotonic() - t_load:.1f}s", flush=True)
    if torch.cuda.is_available():
        used_gb = torch.cuda.memory_allocated() / 1e9
        total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"  VRAM used: {used_gb:.1f} / {total_gb:.1f} GB", flush=True)

    tracker = ProgressTracker(total=len(df))

    for _, row in df.iterrows():
        split_val = str(row["split"])

        if split_val == "none":
            continue
        elif split_val == "clear":
            input_types = ["clear"]
        elif split_val == "vague":
            input_types = ["vague"]
        else:  # "both"
            input_types = ["clear", "vague"]

        if SPLIT_FILTER != "all":
            input_types = [t for t in input_types if t == SPLIT_FILTER]
        if not input_types:
            continue

        try:
            for input_type in input_types:
                row_results = run_multi_turn(agent, row, input_type)
                csv_writer.add(row_results)
            tracker.record(row_results)
        except Exception as exc:
            tracker.record_error()
            print(f"  [ERROR] row={row.name}: {exc}", flush=True)

        if tracker.n_done % SAVE_EVERY == 0:
            csv_writer.flush()
        if tracker.n_done % LOG_EVERY == 0 or tracker.n_done == len(df):
            tracker.print_progress()

    csv_writer.flush()
    # Free VRAM before the next model
    agent.unload()
    if torch.cuda.is_available():
        print(f"  VRAM after unload: {torch.cuda.memory_allocated() / 1e9:.1f} GB", flush=True)
    print(f"  Model {model_name!r} completed.")

elapsed = time.monotonic() - total_t0
print(f"\nAll done in {timedelta(seconds=int(elapsed))}. Output: {OUTPUT_CSV_PATH}")

In [ ]:
res = pd.read_csv(OUTPUT_CSV_PATH)
for col in ("predicted_ids", "planner_concepts"):
    res[col] = res[col].apply(ast.literal_eval)

print(f"Total rows            : {len(res)}")
print(f"\ninput_type distribution:\n{res['input_type'].value_counts()}")

t0 = res[res["turn_pos"] == 0]
print(f"\nTool call rate at turn_pos==0 (by input_type):")
print(t0.groupby("input_type")[["called_get_time","called_get_schedule"]].mean())
print("EXPECTED: ~1.0 for vague, ~0.0 for clear")

print(f"\nplan_method distribution:\n{res['plan_method'].value_counts()}")
print(f"\nresolve_method distribution:\n{res['resolve_method'].value_counts()}")
print(f"\nAverage window size: {res['predicted_ids'].apply(len).mean():.1f}")
print(f"\nFirst 3 rows example:\n{res.head(3).to_string()}")